<h1>Aggregation with dissolve</h1>

<p>Spatial data are often more granular than needed. For example, you might have data on sub-national units, but you’re actually interested in studying patterns at the level of countries.</p>

<p>In a non-spatial setting, when you need summary statistics of the data, you can aggregate data using the
<a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html" title="(in pandas v2.3.0)"><code>groupby()</code></a> function. But for spatial data, you sometimes also need to aggregate geometric features. In the GeoPandas library, you can aggregate geometric features using the <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.dissolve.html" title="geopandas.GeoDataFrame.dissolve"><code>dissolve()</code></a> function.</p>

<p><a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.dissolve.html" title="geopandas.GeoDataFrame.dissolve"><code>dissolve()</code></a> can be thought of as doing three things:</p>

<ol class="loweralpha simple">
<li><p>it dissolves all the geometries within a given group together into a single geometric feature (using the
<a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.union_all.html" title="geopandas.GeoSeries.union_all"><code>union_all()</code></a> method), and</p></li>
<li><p>it aggregates all the rows of data in a group using <a href="https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html#groupby-aggregate" title="(in pandas v2.3.0)">groupby.aggregate</a>, and</p></li>
<li><p>it combines those two results.</p></li>
</ol>

# <h2><a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.dissolve.html" title="geopandas.GeoDataFrame.dissolve"><code>dissolve()</code></a> Example</h2>

<p>Take example of administrative areas in Nepal. You have districts, which are smaller, and zones, which are larger. A group of districts always compose a single zone. Suppose you are interested in Nepalese zone, but you only have Nepalese district-level data like the <cite>geoda.nepal</cite> dataset included in <cite>geodatasets</cite>. You can easily convert this to a zone-level dataset.</p>

<p>First, let’s look at the most simple case where you just want zone shapes and names.</p>

In [1]:
!pip install geopandas
!pip install geodatasets

In [2]:
import geopandas
import geodatasets

In [3]:
nepal = geopandas.read_file(geodatasets.get_path('geoda.nepal'))
nepal = nepal.rename(columns={"name_2": "zone"})

In [4]:
nepal[["zone", "geometry"]].head()

,zone,geometry
0,Dhaualagiri,"POLYGON ((83.10834 28.6202, 83.1056 28.60976, ..."
1,Dhaualagiri,"POLYGON ((83.99726 29.31675, 84 29.31576, 84 2..."
2,Dhaualagiri,"POLYGON ((83.50688 28.79306, 83.51024 28.78809..."
3,Dhaualagiri,"POLYGON ((83.70261 28.39837, 83.70435 28.39452..."
4,Bagmati,"POLYGON ((85.52173 27.71822, 85.52359 27.71375..."


<p>By default, <a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.dissolve.html" title="geopandas.GeoDataFrame.dissolve"><code>dissolve()</code></a> will pass <code>'first'</code> to <a href="https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html#groupby-aggregate" title="(in pandas v2.3.0)">groupby.aggregate</a>.</p>

In [5]:
nepal_zone = nepal[["zone", "geometry"]]
zones = nepal_zone.dissolve(by='zone')

In [6]:
zones.head()

,geometry
zone,
Bagmati,"POLYGON ((85.87653 27.61234, 85.87355 27.60861..."
Bheri,"POLYGON ((81.75089 28.31038, 81.75562 28.3074,..."
Dhaualagiri,"POLYGON ((83.70647 28.39278, 83.70721 28.38781..."
Gandaki,"POLYGON ((84.49995 28.74099, 84.50443 28.7441,..."
Janakpur,"POLYGON ((86.26166 26.91417, 86.2588 26.91144,..."


In [8]:
import plotly.express as px

geo_df = zones

fig = px.choropleth(geo_df, geojson=geo_df.geometry, locations=geo_df.index)
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(margin={'r':0,'t':0,'l':0,'b':0})
fig.show()

<p>If you are interested in aggregate populations, however, you can pass different functions to the
<a href="https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.dissolve.html" title="geopandas.GeoDataFrame.dissolve"><code>dissolve()</code></a> method to aggregate populations using the <code>aggfunc =</code> argument:</p>

In [13]:
nepal_pop = nepal[['zone', 'geometry', 'population']]
zones = nepal_pop.dissolve(by='zone', aggfunc='sum')

In [14]:
zones.head()

,geometry,population
zone,,
Bagmati,"POLYGON ((85.87653 27.61234, 85.87355 27.60861...",3750441
Bheri,"POLYGON ((81.75089 28.31038, 81.75562 28.3074,...",1463510
Dhaualagiri,"POLYGON ((83.70647 28.39278, 83.70721 28.38781...",516905
Gandaki,"POLYGON ((84.49995 28.74099, 84.50443 28.7441,...",1530310
Janakpur,"POLYGON ((86.26166 26.91417, 86.2588 26.91144,...",2818356


In [19]:
import plotly.express as px

geo_df = zones

fig = px.choropleth(geo_df, geojson=geo_df.geometry, locations=geo_df.index, color=geo_df['population'], color_continuous_scale='YlOrRd')
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(margin={'r':0,'t':0,'l':0,'b':0})
fig.update_layout(coloraxis_showscale=False)
fig.show()

# <h2>Dissolve arguments</h2>

<p>The <code>aggfunc =</code> argument defaults to ‘first’ which means that the first row of attributes values found in the dissolve routine will be assigned to the resultant dissolved geodataframe. However it also accepts other summary statistic options as allowed by
<a href="https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html" title="(in pandas v2.3.0)"><code>pandas.groupby</code></a> including:</p>

<ul class="simple">
<li><p>‘first’</p></li>
<li><p>‘last’</p></li>
<li><p>‘min’</p></li>
<li><p>‘max’</p></li>
<li><p>‘sum’</p></li>
<li><p>‘mean’</p></li>
<li><p>‘median’</p></li>
<li><p>function</p></li>
<li><p>string function name</p></li>
<li><p>list of functions and/or function names, e.g. [np.sum, ‘mean’]</p></li>
<li><p>dict of axis labels -&gt; functions, function names or list of such.</p></li>
</ul>

<p>For example, to get the number of countries on each continent, as well as the populations of the largest and smallest country of each,
you can aggregate the <code>'name'</code> column using <code>'count'</code>, and the <code>'pop_est'</code> column using <code>'min'</code> and <code>'max'</code>:</p>

In [ ]:
zones = nepal.dissolve(
     by="zone",
     aggfunc={
         "district": "count",
         "population": ["min", "max"],
     },
 )
zones.head()

,geometry,"(district, count)","(population, min)","(population, max)"
zone,,,,
Bagmati,"POLYGON ((85.87653 27.61234, 85.87355 27.60861...",8,42125,1688131
Bheri,"POLYGON ((81.75089 28.31038, 81.75562 28.30740...",5,170090,422812
Dhaualagiri,"POLYGON ((83.70647 28.39278, 83.70721 28.38781...",4,11585,250065
Gandaki,"POLYGON ((84.49995 28.74099, 84.50443 28.74410...",6,5827,480851
Janakpur,"POLYGON ((86.26166 26.91417, 86.25880 26.91144...",6,184931,765959
